# Make chapters for the scroll reader

Turns your downloaded chapters (CBZ files) into ready-to-direct chapters: it drops the title and promo pages,
splits every panel into a background, a character layer and speech bubbles, converts it all for the web,
and makes overview sheets. Each chapter ends up as **one zip of about 15 MB** to send to Claude.

**First time only:** in [Google Drive](https://drive.google.com) make a folder **`Storyteller`**, and inside it a folder **`cbz`**.
Put your CBZ files in `Storyteller/cbz` (you can drag the whole folder of files into the browser).

**Every time**
1. Menu **Runtime → Change runtime type → T4 GPU**, and switch **High-RAM** on.
2. In Step 1 below, type which chapters to make.
3. Menu **Runtime → Run all**. When Google asks, allow access to your Google Drive.
4. The finished zips land in **`Storyteller/chapters`** in your Drive (and download to your computer too). Send them to Claude.

About **15 minutes per chapter**. If Colab disconnects, just **Run all** again: finished chapters are skipped.

### Step 1: which chapters

In [ ]:
CHAPTERS = "94"        # one chapter "94", a range "1-5", or a list "1-5, 94"
SKIP_PAGES = {}        # pages to drop by hand, e.g. {94: [0, 69]} (the title and promo pages are dropped automatically)
REDO = False           # True: make chapters again even if they are already done
DOWNLOAD = True        # True: also download the zips to this computer when they are done
DRIVE_FOLDER = "Storyteller"   # your folder in Google Drive

### Step 2: connect Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
ROOT = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
CBZ, OUT = f'{ROOT}/cbz', f'{ROOT}/chapters'
if not os.path.isdir(CBZ):
    raise SystemExit(f'Make a folder "{DRIVE_FOLDER}/cbz" in your Google Drive and put the CBZ files in it, then Run all again.')
os.makedirs(OUT, exist_ok=True)
count = sum(f.lower().endswith('.cbz') for _, _, names in os.walk(CBZ) for f in names)
print(f"{count} CBZ files in {DRIVE_FOLDER}/cbz")
if count == 0:
    raise SystemExit(f'No CBZ files found in "{DRIVE_FOLDER}/cbz" in your Google Drive. Put them there, then Run all again.')

### Step 3: install the tools (about 1 minute)
If a red pip message about "dependency conflicts" shows up, it is harmless.

In [ ]:
# GPU build of the cutout runtime, so the cutout models run on the T4 graphics card
# (it still works without one, just slower). The plain build must go first: both cannot be installed.
!pip -q uninstall -y onnxruntime 2>/dev/null
!pip -q install "rembg[gpu]" opencv-python-headless scipy
# --no-deps: this package pins an old Pillow that no longer installs on Colab;
# Colab already has everything it needs (torch, Pillow, numpy, OpenCV).
!pip -q install --no-deps simple-lama-inpainting==0.1.2

### Step 4: the two scripts (these cells just save them)

In [ ]:
%%writefile split_layers.py
"""Split manga / webtoon strips into scroll-ready layers.

For every image you give it, this script:
  1. finds each panel on the strip (the boxes separated by black or white gutters),
  2. finds the speech bubbles and saves each one as a see-through PNG,
  3. cuts the characters out of each panel as a see-through PNG, with clean edges,
  4. paints over the holes the characters and bubbles leave, so the panel
     background looks complete when the layers move apart on scroll.

Usage:
    python split_layers.py input_folder_or_image [more ...] --out output_folder

Everything lands in output_folder/<image name>/ together with a layers.json
that says where each layer sits on the original strip.

If your images are slices of one long webtoon strip (so a panel can be cut in
two between images), add --join to stack them back together first.
"""

import argparse
import json
import re
import sys
import time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
from PIL import Image
from scipy import ndimage as ndi

IMAGE_TYPES = {".png", ".jpg", ".jpeg", ".webp"}
# Picked by comparing all of rembg's models on real webtoon panels: general is the most
# reliable, hrsod adds missed edges only, while lite and massive grab chunks of background.
CUTOUT_MODELS = ["birefnet-general", "birefnet-hrsod"]


# ---------------------------------------------------------------- bubbles

def find_bubbles(rgb):
    """Return one (y, x, mask) per speech bubble: the filled mask of its box and where that box sits.

    A bubble is a near-white area with dark lettering inside it. Large white
    areas without lettering (glows, white skies) are skipped, and so are white
    slashes and highlights whose "holes" are coloured effects rather than black ink.
    """
    hi = rgb.max(2)
    lo = rgb.min(2)
    white = (lo > 225) & (hi - lo < 25)
    labels, _ = ndi.label(white)
    bubbles = []
    for i, box in enumerate(ndi.find_objects(labels), 1):
        part = labels[box] == i
        area = part.sum()
        if area < 1500:
            continue
        filled = ndi.binary_fill_holes(part)
        filled_area = filled.sum()
        lettering = (filled_area - area) / filled_area
        roundness = filled_area / part.size
        if not (0.02 < lettering < 0.45 and roundness > 0.45):
            continue
        # lettering is black ink; real bubbles score about 0.65 here, slashes and glows about 0
        holes = filled & ~part
        ink = (hi[box][holes] < 100).mean()
        if ink <= 0.3:
            continue
        # lettering is several separate dark shapes in neutral ink. An eye's white has one dark shape
        # (the pupil), and a white face (Sylvie's fox form) has coloured eyes: both are left in the art.
        dark = holes & (hi[box] < 110)
        letters, n = ndi.label(dark)
        shapes = 0
        for s, piece in enumerate(ndi.find_objects(letters), 1):
            height = piece[0].stop - piece[0].start
            if 5 <= height <= 70 and (letters[piece] == s).sum() >= 6:
                shapes += 1
        colourful = ((hi[box].astype(np.int16) - lo[box]) > 60)[holes].mean()
        if shapes >= 2 and colourful <= 0.05:
            bubbles.append((box[0].start, box[1].start, filled))
    return bubbles


# ---------------------------------------------------------------- panels

def _runs(flags, min_len):
    out, start = [], None
    for i, on in enumerate(list(flags) + [False]):
        if on and start is None:
            start = i
        if not on and start is not None:
            if i - start >= min_len:
                out.append((start, i))
            start = None
    return out


def find_panels(rgb, bubble_mask):
    """Return panel boxes as (x, y, w, h), top to bottom.

    Works on vertical webtoon strips where panels are stacked with gutters
    between them. The gutter colour is the most common colour on the image border.
    """
    border = np.concatenate([rgb[0], rgb[-1], rgb[:, 0], rgb[:, -1]]) // 16
    colours, counts = np.unique(border, axis=0, return_counts=True)
    gutter_colour = colours[counts.argmax()].astype(np.int16) * 16 + 8
    # int16 rather than int, so a whole joined chapter fits in memory
    distance = np.abs(rgb.astype(np.int16) - gutter_colour).max(2)
    ignore = cv2.dilate(bubble_mask.astype(np.uint8), np.ones((15, 15), np.uint8)) > 0
    content = (distance > 18) & ~ignore

    panels = []
    rows = content.mean(1) > 0.05
    for y0, y1 in _runs(rows, 60):
        cols = content[y0:y1].mean(0) > 0.05
        col_runs = _runs(cols, 30)
        if not col_runs:
            continue
        x0, x1 = col_runs[0][0], col_runs[-1][1]
        panels.append((x0, y0, x1 - x0, y1 - y0))
    if not panels:  # a single picture with no gutters
        h, w = rgb.shape[:2]
        panels.append((0, 0, w, h))
    return panels


# ---------------------------------------------------------------- models

class Models:
    def __init__(self, cutout_models=None):
        # torch first: the GPU build of onnxruntime runs on the CUDA libraries torch loads
        import torch
        import onnxruntime
        from rembg.sessions import sessions_class
        from simple_lama_inpainting.models.model import LAMA_MODEL_URL
        from simple_lama_inpainting.utils.util import download_model

        if hasattr(onnxruntime, "preload_dlls"):  # Windows: find the CUDA libraries that came with torch
            try:
                onnxruntime.preload_dlls()
            except Exception:
                pass
        providers = onnxruntime.get_available_providers()
        # NVIDIA's CUDA where it works (Colab, Kaggle); DirectML on Windows works with any graphics card
        self.gpu_kind = ("cuda" if torch.cuda.is_available() and "CUDAExecutionProvider" in providers
                         else "dml" if "DmlExecutionProvider" in providers else None)
        gpu = self.gpu_kind is not None
        # seconds spent in each step, so a run shows where its time goes
        self.timings = defaultdict(float)
        if cutout_models is None:
            # the second model only adds edge pixels; on the processor it is not worth
            # the time, and both together need more memory than Colab has (about 14 GB)
            cutout_models = CUTOUT_MODELS if gpu else CUTOUT_MODELS[:1]
        print(f"Loading cutout model(s) {', '.join(cutout_models)} and background fill model ...")
        # lower-memory settings so the bigger models also fit on an ordinary computer
        options = onnxruntime.SessionOptions()
        options.enable_cpu_mem_arena = False
        options.enable_mem_pattern = False
        # made directly rather than with rembg's new_session(), which ignores these settings
        # before rembg 2.0.78; without them the two models need about 14 GB and crash Colab
        by_name = {c.name(): c for c in sessions_class}
        unknown = [name for name in cutout_models if name not in by_name]
        if unknown:
            raise SystemExit(f"Unknown cutout model(s): {', '.join(unknown)}")
        # made on the processor, then moved to the graphics card below once they are known to run there
        self.cutouts = [by_name[name](name, options, providers=["CPUExecutionProvider"]) for name in cutout_models]
        # Loaded by hand instead of SimpleLama() so it also works without a GPU.
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.lama = torch.jit.load(download_model(LAMA_MODEL_URL), map_location=self.device).eval()
        self.max_fill_size = 2048 if self.device.type == "cuda" else 800
        try:  # edge refinement arrived in rembg 2.0.8x; older versions just skip it
            from rembg.matting import vitmatte_alpha
            self.refine_edges = vitmatte_alpha
        except ImportError:
            self.refine_edges = None

        noise = Image.fromarray(np.random.default_rng(0).integers(0, 256, (64, 64, 3), np.uint8))
        cutouts_on_gpu = gpu and all([self._move_to_gpu(c.inner_session, lambda c=c: c.predict(noise), self.gpu_kind)
                                      for c in self.cutouts])
        if cutouts_on_gpu and self.refine_edges:
            try:  # rembg keeps its ViTMatte session to itself; it trips over the same problem
                from rembg import matting
                self._move_to_gpu(matting._get_session(matting.DEFAULT_VARIANT),
                                  lambda: self.refine_edges(noise, noise.convert("L")), self.gpu_kind)
            except Exception:
                pass
        self.cutouts_on_gpu = cutouts_on_gpu

        where = lambda on_gpu: "graphics card (fast)" if on_gpu else "processor (slow)"
        print(f"Cutouts run on the {where(cutouts_on_gpu)}, background fill on the {where(self.device.type == 'cuda')}.")
        if torch.cuda.is_available():
            print(f"Graphics card: {torch.cuda.get_device_name(0)}" + (" (cutouts through DirectML)" if self.gpu_kind == "dml" else ""))
        in_colab = "google.colab" in sys.modules
        if not torch.cuda.is_available() and in_colab:
            print("No graphics card found. For a much faster run choose Runtime > Change runtime type > T4 GPU.")
        elif not cutouts_on_gpu and in_colab:
            print("The cutout models could not use this graphics card. Try Runtime > Change runtime type > L4 GPU.")

    @staticmethod
    def _move_to_gpu(session, warm_up, kind="cuda"):
        """Run an onnxruntime session on the graphics card if it works there; say whether it does.

        onnxruntime's default way of picking convolution algorithms (EXHAUSTIVE) fails on a T4
        with the cuDNN that comes with CUDA 13, and then quietly falls back to the processor.
        So try the other two ways, checking each with a warm-up run.
        """
        import onnxruntime

        session.disable_fallback()  # fail loudly here instead of silently moving to the processor
        if kind == "dml":  # DirectML (Windows): no algorithm choice and no memory pool to manage
            try:
                session.set_providers(["DmlExecutionProvider", "CPUExecutionProvider"])
                warm_up()
                return "DmlExecutionProvider" in session.get_providers()
            except Exception:
                session.set_providers(["CPUExecutionProvider"])
                return False
        onnxruntime.set_default_logger_severity(4)  # a failed try prints pages of cuDNN errors; expected here
        try:
            for search in ("HEURISTIC", "DEFAULT"):  # on a T4 only DEFAULT works; newer cards take HEURISTIC
                try:
                    session.set_providers(["CUDAExecutionProvider", "CPUExecutionProvider"],
                                          [{"cudnn_conv_algo_search": search, "arena_extend_strategy": "kSameAsRequested"}, {}])
                    warm_up()
                    if "CUDAExecutionProvider" in session.get_providers():
                        break
                except Exception:
                    pass
            else:
                session.set_providers(["CPUExecutionProvider"])
                return False
        finally:
            onnxruntime.set_default_logger_severity(2)

        # By default each session keeps all the graphics memory it ever used, and the three of
        # them filled a 15 GB T4 before the background fill got any. Hand it back after every run.
        shrink = onnxruntime.RunOptions()
        shrink.add_run_config_entry("memory.enable_memory_arena_shrinkage", "gpu:0")
        run = session.run
        session.run = lambda output_names, input_feed, run_options=None: run(output_names, input_feed, shrink)
        try:
            warm_up()
        except Exception:  # onnxruntime refuses it when there is no graphics memory pool to shrink
            del session.run
        return True

    def character_alpha(self, rgb):
        image = Image.fromarray(rgb)
        mirrored = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        masks = []
        for session in self.cutouts:
            started = time.perf_counter()
            # averaging with the mirrored panel drops glows and auras only one view mistakes for a character
            with np.errstate(over="ignore"):  # rembg's sigmoid overflows to exactly 0 on sure background, which is right
                plain = np.asarray(session.predict(image)[0], np.float32)
                flipped = np.asarray(session.predict(mirrored)[0], np.float32)[:, ::-1]
            masks.append((plain + flipped) / 2)
            self.timings[f"cutout {session.model_name}"] += time.perf_counter() - started
        # each model misses different bits (a hand here, an outline there),
        # so keep anything that any of them sees as part of a character
        alpha = np.maximum.reduce(masks).astype(np.uint8)
        if self.refine_edges:
            started = time.perf_counter()
            # ViTMatte redraws the soft edge band: crisper hair, no see-through ghost patches
            refined = np.asarray(self.refine_edges(image, Image.fromarray(alpha)))
            self.timings["edge refinement"] += time.perf_counter() - started
            # its output tops out at 254; make the solid inside fully solid again
            refined = np.where(refined >= 250, 255, refined)
            # it also thins fine lines the models were sure of (speed lines, outlines), so those keep their value
            alpha = np.maximum(refined, np.where(alpha >= 200, alpha, 0)).astype(np.uint8)
        return alpha.copy()

    def character_colours(self, rgb, alpha):
        from pymatting import estimate_foreground_ml

        started = time.perf_counter()
        # on soft edges the drawn colour is mixed with the background behind it;
        # unmix it so the character carries no fringe of the old background
        foreground = estimate_foreground_ml(rgb / 255.0, alpha / 255.0)
        self.timings["edge colours"] += time.perf_counter() - started
        foreground = np.clip(foreground * 255 + 0.5, 0, 255).astype(np.uint8)
        return np.where(alpha[..., None] == 255, rgb, foreground)

    def fill_background(self, rgb, hole_mask):
        if not hole_mask.any():
            return rgb
        import torch
        from simple_lama_inpainting.utils.util import prepare_img_and_mask

        started = time.perf_counter()
        h, w = rgb.shape[:2]
        fill_size = min(self.max_fill_size, max(h, w))
        while True:
            # big panels are filled at a smaller size to save memory, then scaled back
            scale = min(1.0, fill_size / max(h, w))
            small = (round(w * scale), round(h * scale))
            image = Image.fromarray(rgb).resize(small, Image.LANCZOS)
            mask = Image.fromarray(hole_mask.astype(np.uint8) * 255).resize(small, Image.NEAREST)
            image, mask = prepare_img_and_mask(image, mask, self.device)
            try:
                with torch.inference_mode():
                    result = self.lama(image, mask)[0].permute(1, 2, 0).cpu().numpy()
                break
            except RuntimeError as error:  # the traced model reports a full graphics card as a plain RuntimeError
                if "out of memory" not in str(error) or fill_size <= 256:
                    raise
                del image, mask
                torch.cuda.empty_cache()
                fill_size //= 2
        if self.device.type == "cuda":
            torch.cuda.empty_cache()  # leave the graphics memory to the cutout models
        self.timings["background fill"] += time.perf_counter() - started
        result = np.clip(result * 255, 0, 255).astype(np.uint8)[:small[1], :small[0]]
        result = cv2.resize(result, (w, h), interpolation=cv2.INTER_CUBIC)
        # keep the untouched pixels exactly as drawn
        return np.where(hole_mask[..., None], result, rgb)


# ---------------------------------------------------------------- main work

def save_rgba(path, rgb, alpha):
    Image.fromarray(np.dstack([rgb, alpha]).astype(np.uint8), "RGBA").save(path)


def natural_key(path):
    # so 2.png comes before 10.png
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", path.name)]


def join_slices(files):
    images = [np.array(Image.open(p).convert("RGB")) for p in files]
    widths = sorted({im.shape[1] for im in images})
    if len(widths) > 1:
        raise SystemExit(f"--join needs images that are all the same width, got widths {widths}.")
    return np.vstack(images)


def split_image(name, source, rgb, out_root, models):
    h, w = rgb.shape[:2]
    out = out_root / name
    out.mkdir(parents=True, exist_ok=True)
    grow = lambda m, px: cv2.dilate(m.astype(np.uint8), np.ones((px, px), np.uint8)) > 0

    def grow_bubble(y, x, mask, px):
        # grow within the bubble's box plus a margin, not a full-strip mask per bubble
        pad = px // 2
        y0, x0 = max(y - pad, 0), max(x - pad, 0)
        y1, x1 = min(y + mask.shape[0] + pad, h), min(x + mask.shape[1] + pad, w)
        box = np.zeros((y1 - y0, x1 - x0), bool)
        box[y - y0:y - y0 + mask.shape[0], x - x0:x - x0 + mask.shape[1]] = mask
        return y0, x0, grow(box, px)

    # grow each bubble a little so its black outline comes along with it
    bubbles = [grow_bubble(*b, 15) for b in find_bubbles(rgb)]
    all_bubbles = np.zeros((h, w), bool)
    for y, x, mask in bubbles:
        all_bubbles[y:y + mask.shape[0], x:x + mask.shape[1]] |= mask
    panels = find_panels(rgb, all_bubbles)

    info = {"source": source, "width": w, "height": h, "panels": [], "bubbles": []}
    preview = rgb.copy()

    for n, (x, y, pw, ph) in enumerate(panels, 1):
        name = f"panel_{n:02d}"
        folder = out / name
        folder.mkdir(exist_ok=True)
        crop = rgb[y:y + ph, x:x + pw]
        bubble_here = all_bubbles[y:y + ph, x:x + pw]

        alpha = models.character_alpha(crop)
        alpha[grow(bubble_here, 15)] = 0
        alpha[alpha < 20] = 0
        # grow the holes so glows and outlines around characters get painted over too
        holes = grow(alpha > 40, 31) | grow(bubble_here, 9)
        background = models.fill_background(crop, holes)

        Image.fromarray(crop).save(folder / "original.png")
        Image.fromarray(background).save(folder / "background.png")
        save_rgba(folder / "characters.png", models.character_colours(crop, alpha), alpha)

        info["panels"].append({
            "id": name, "x": int(x), "y": int(y), "width": int(pw), "height": int(ph),
            "layers": {
                "background": f"{name}/background.png",
                "characters": f"{name}/characters.png",
                "original": f"{name}/original.png",
            },
        })
        cv2.rectangle(preview, (x, y), (x + pw, y + ph), (0, 255, 0), 4)
        cut = (alpha > 40)
        region = preview[y:y + ph, x:x + pw]
        region[cut] = (region[cut] * 0.5 + np.array([255, 0, 255]) * 0.5).astype(np.uint8)
        print(f"  {name}: {pw}x{ph} at ({x}, {y})")

    (out / "bubbles").mkdir(exist_ok=True)
    for n, (by, bx, mask) in enumerate(bubbles, 1):
        ys, xs = np.nonzero(mask)
        mask = mask[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        y0, x0 = by + ys.min(), bx + xs.min()
        y1, x1 = y0 + mask.shape[0], x0 + mask.shape[1]
        file = f"bubbles/bubble_{n:02d}.png"
        save_rgba(out / file, rgb[y0:y1, x0:x1], mask * 255)
        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
        owner = next((p["id"] for p in info["panels"]
                      if p["x"] <= cx <= p["x"] + p["width"] and p["y"] <= cy <= p["y"] + p["height"]), None)
        info["bubbles"].append({"id": f"bubble_{n:02d}", "x": int(x0), "y": int(y0),
                                "width": int(x1 - x0), "height": int(y1 - y0),
                                "panel": owner, "file": file})
        region = preview[y0:y1, x0:x1]
        region[mask] = (region[mask] * 0.4 + np.array([255, 60, 60]) * 0.6).astype(np.uint8)
    print(f"  {len(bubbles)} speech bubble(s)")

    Image.fromarray(preview).save(out / "preview.png")
    (out / "layers.json").write_text(json.dumps(info, indent=2))


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("inputs", nargs="+", help="image files or folders of images")
    parser.add_argument("--out", default="layers", help="where to put the results (default: layers)")
    parser.add_argument("--join", action="store_true",
                        help="stack all images into one long strip first (for webtoon slices "
                             "where panels are cut in two between images)")
    parser.add_argument("--models",
                        help="comma-separated cutout models to combine (default: "
                             f"{','.join(CUTOUT_MODELS)} with a graphics card, {CUTOUT_MODELS[0]} without); "
                             "use just birefnet-general-lite for a quick, rougher run")
    args = parser.parse_args()

    files = []
    for item in map(Path, args.inputs):
        if item.is_dir():
            files += sorted((p for p in item.iterdir() if p.suffix.lower() in IMAGE_TYPES), key=natural_key)
        else:
            files.append(item)
    if not files:
        raise SystemExit("No images found.")

    models = Models(args.models.split(",") if args.models else None)
    out_root = Path(args.out)
    if args.join:
        name = f"{files[0].stem}-{files[-1].stem}" if len(files) > 1 else files[0].stem
        print(f"{name} (joined {len(files)} images)")
        split_image(name, [p.name for p in files], join_slices(files), out_root, models)
    else:
        for path in files:
            print(f"{path.name}")
            split_image(path.stem, path.name, np.array(Image.open(path).convert("RGB")), out_root, models)
    print(f"Done. Layers are in {out_root.resolve()}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile make_chapter.py
"""Turn downloaded chapters (CBZ files) into ready-to-direct chapters for the scroll reader.

For each chapter it:
  1. finds the chapter's CBZ by its number ("..._94_-_Chapter_94.cbz" is chapter 94),
  2. takes out the pages in order and drops the series title page and the creators' promo page
     (recognised by their look; pages can also be dropped by hand with --skip),
  3. joins the pages into one long strip and splits it into panels, characters, bubbles and filled
     backgrounds with split_layers.py,
  4. converts everything to WebP for the web, works out which panel each speech bubble belongs to,
     and draws overview sheets of the pages, panels and bubbles for the director,
  5. saves the result as <out>/chapter_<id>.zip (about 7-10 MB): the only file it writes to <out>.

Chapters already done are skipped, so a batch can be restarted after Colab disconnects.

Usage:
    python make_chapter.py --cbz-dir CBZ_FOLDER --out OUT_FOLDER --chapters 1-5,94 [--skip '{"94": [0, 69]}'] [--redo]
                           [--shard 1/2] [--done-dir FOLDER] [--max-hours 11]

--shard K/N makes this run take every Nth chapter of the batch starting at the Kth, so N graphics cards can share it
(the split is the same whatever is already done); --done-dir skips chapters whose zip is already in another folder
(an earlier run's output); --max-hours stops starting new chapters after that long.
"""

import argparse
import json
import re
import shutil
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw

Image.MAX_IMAGE_PIXELS = None  # a joined chapter is one very tall picture

IMAGE_TYPES = {".png", ".jpg", ".jpeg", ".webp", ".avif"}
WEBP = {"background": 80, "characters": 85, "bubble": 90}

# Fingerprints (difference hashes, 16 x 32) of pages that are not part of the story, taken from chapter 94.
# Only the first two and last two pages of a chapter are compared with them.
NON_STORY = {
    "series title page": "00000000040040000d004ca064a07d70b4647cf82d0c1c003260b4f0347003000f9006800cc00ec0030007000600000000000000000000000000c0000000f634",
    "creators' promo page": "000000000000000000000000000000000000000000000000000000002200000004001900268043404f40d901191c0d634f332a349a9b42780160000000000000",
}
MATCH_BITS = 110  # of 512; the closest story page in chapter 94 differs from these by 156


# ---------------------------------------------------------------- chapters and pages

def chapter_number(path):
    """'The_Beginning_After_the_End_94_-_Chapter_94.cbz' -> '94' (also '94.5'); None if there is no number."""
    m = re.search(r"chapter[\s_-]*(\d+(?:\.\d+)?)", path.stem, re.I) or re.search(r"(\d+(?:\.\d+)?)(?!.*\d)", path.stem)
    if not m:
        return None
    whole, _, part = m.group(1).partition(".")
    return str(int(whole)) + (f".{part}" if part else "")


def chapter_id(number):
    """'94' -> '094', '94.5' -> '094-5': the folder name and the reader's #address."""
    whole, _, part = number.partition(".")
    return whole.zfill(3) + (f"-{part}" if part else "")


def parse_chapters(text):
    """'1-5,94,100.5' -> a test that says whether a chapter number is wanted."""
    ranges = []
    for part in filter(None, (p.strip() for p in text.split(","))):
        a, _, b = part.partition("-")
        ranges.append((float(a), float(b or a)))
    return lambda n: any(a <= float(n) <= b for a, b in ranges)


def natural_key(name):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", name)]


def extract_pages(cbz, into):
    into.mkdir(parents=True, exist_ok=True)
    pages = []
    with zipfile.ZipFile(cbz) as z:
        names = [n for n in z.namelist()
                 if Path(n).suffix.lower() in IMAGE_TYPES and not n.startswith("__MACOSX") and not n.endswith("/")]
        for k, name in enumerate(sorted(names, key=natural_key)):
            target = into / f"{k:03d}{Path(name).suffix.lower()}"
            target.write_bytes(z.read(name))
            pages.append(target)
    return pages


def dhash(image, w=16, h=32):
    g = np.asarray(image.convert("L").resize((w + 1, h), Image.LANCZOS), dtype=np.int16)
    return (g[:, 1:] > g[:, :-1]).flatten()


REFS = {why: np.unpackbits(np.frombuffer(bytes.fromhex(h), dtype=np.uint8)).astype(bool) for why, h in NON_STORY.items()}


def non_story_reason(image):
    h = dhash(image)
    for why, ref in REFS.items():
        if int((h != ref).sum()) <= MATCH_BITS:
            return why
    return None


# ---------------------------------------------------------------- overview sheets for the director

def label(draw, xy, text):
    x, y = xy
    draw.rectangle([x, y, x + 7 * len(text) + 8, y + 16], fill=(0, 0, 0))
    draw.text((x + 4, y + 2), text, fill=(255, 230, 0))


def pages_sheet(images, dropped, path):
    """Every page side by side, dropped pages crossed out in red."""
    H, cols = 330, 14
    thumbs = [im.convert("RGB").resize((max(1, round(im.width * H / im.height)), H)) for im in images]
    w = max(t.width for t in thumbs)
    sheet = Image.new("RGB", (cols * (w + 4), -(-len(thumbs) // cols) * (H + 20)), (110, 110, 110))
    d = ImageDraw.Draw(sheet)
    for k, t in enumerate(thumbs):
        x, y = (k % cols) * (w + 4), (k // cols) * (H + 20)
        sheet.paste(t, (x, y + 18))
        label(d, (x, y), f"{k:03d}" + (" DROPPED" if k in dropped else ""))
        if k in dropped:
            d.line([x, y + 18, x + t.width, y + 18 + H], fill=(230, 30, 40), width=5)
            d.line([x + t.width, y + 18, x, y + 18 + H], fill=(230, 30, 40), width=5)
    sheet.save(path, quality=80)


def panel_sheets(panel_images, folder, per_sheet=5, height=900):
    """Original panels, five per sheet, labelled with their number and size."""
    paths = []
    for s in range(0, len(panel_images), per_sheet):
        group = panel_images[s:s + per_sheet]
        thumbs = []
        for n, im in group:
            k = min(height / im.height, 420 / im.width)
            thumbs.append((n, im.size, im.convert("RGB").resize((max(1, round(im.width * k)), max(1, round(im.height * k))))))
        sheet = Image.new("RGB", (sum(t.width + 10 for _, _, t in thumbs), height + 24), (110, 110, 110))
        d = ImageDraw.Draw(sheet)
        x = 0
        for n, size, t in thumbs:
            sheet.paste(t, (x, 24))
            label(d, (x, 2), f"panel {n:02d}  {size[0]}x{size[1]}" + ("  TINY" if min(size) < 120 or size[0] * size[1] < 40000 else ""))
            x += t.width + 10
        path = folder / f"panels_{group[0][0]:02d}-{group[-1][0]:02d}.jpg"
        sheet.save(path, quality=82)
        paths.append(path)
    return paths


def bubbles_sheet(bubbles, path, height=240):
    """Every bubble on grey, labelled with its number and the panel it belongs to."""
    if not bubbles:
        return
    thumbs = []
    for b, im in bubbles:
        k = height / im.height
        bg = Image.new("RGBA", im.size, (90, 90, 90, 255))
        bg.alpha_composite(im.convert("RGBA"))
        thumbs.append((b, bg.convert("RGB").resize((max(1, round(im.width * k)), height))))
    rows, row, width = [], [], 0
    for item in thumbs:
        if row and width + item[1].width > 2400:
            rows.append(row)
            row, width = [], 0
        row.append(item)
        width += item[1].width + 8
    rows.append(row)
    sheet = Image.new("RGB", (max(sum(t.width + 8 for _, t in r) for r in rows), len(rows) * (height + 24)), (0, 0, 0))
    d = ImageDraw.Draw(sheet)
    for r, items in enumerate(rows):
        x = 0
        for b, t in items:
            sheet.paste(t, (x, r * (height + 24) + 22))
            label(d, (x, r * (height + 24) + 2), f"{b['id']} > {b['panel']}" + (" ?" if b["suspect"] else ""))
            x += t.width + 8
    sheet.save(path, quality=85)


# ---------------------------------------------------------------- one chapter

def make_chapter(cbz, number, out_root, work_root, models, skip_by_hand):
    import split_layers  # sits next to this file

    cid = chapter_id(number)
    work = work_root / cid
    shutil.rmtree(work, ignore_errors=True)
    started = time.time()

    # 1-2. pages, minus the non-story ones
    files = extract_pages(cbz, work / "pages")
    if not files:
        raise RuntimeError(f"{cbz.name} has no images in it.")
    images = [Image.open(p) for p in files]
    dropped = {}
    for k in sorted(set(range(min(2, len(images)))) | set(range(max(0, len(images) - 2), len(images)))):
        why = non_story_reason(images[k])
        if why:
            dropped[k] = why
    for k in skip_by_hand:
        dropped[k if k >= 0 else len(images) + k] = "dropped by hand"
    kept = [k for k in range(len(images)) if k not in dropped]
    print(f"  {len(images)} pages; dropped: " + (", ".join(f"{k:03d} ({why})" for k, why in sorted(dropped.items())) or "none"))

    # 3. join and split. Pages of a different width (rare) are scaled to the most common one.
    widths = [images[k].width for k in kept]
    width = max(set(widths), key=widths.count)
    strip = []
    for k in kept:
        im = images[k].convert("RGB")
        if im.width != width:
            print(f"  page {k:03d} is {im.width} px wide, scaled to {width}")
            im = im.resize((width, round(im.height * width / im.width)), Image.LANCZOS)
        strip.append(np.asarray(im))
    rgb = np.vstack(strip)
    del strip
    print(f"  joined strip: {rgb.shape[1]} x {rgb.shape[0]}")
    before = dict(getattr(models, "timings", {}))
    split_started = time.time()
    split_layers.split_image(cid, [files[k].name for k in kept], rgb, work / "layers", models)
    split_seconds = time.time() - split_started
    del rgb
    steps = {k: round(v - before.get(k, 0), 1) for k, v in getattr(models, "timings", {}).items() if v - before.get(k, 0) > 0}
    layers = work / "layers" / cid
    info = json.loads((layers / "layers.json").read_text())

    # 4. package for the web (in the scratch folder: only the zip goes to <out>, which may be Google Drive)
    out = work / "package" / cid
    (out / "sheets").mkdir(parents=True)
    panels, panel_images = [], []
    for n, p in enumerate(info["panels"], 1):
        folder = layers / p["id"]
        Image.open(folder / "background.png").convert("RGB").save(
            out / f"{p['id']}_background.webp", quality=WEBP["background"], method=6)
        Image.open(folder / "characters.png").convert("RGBA").save(
            out / f"{p['id']}_characters.webp", quality=WEBP["characters"], method=6)
        panel_images.append((n, Image.open(folder / "original.png")))
        entry = {k: p[k] for k in ("id", "x", "y", "width", "height")}
        # crumbs the panel finder picks up (a flash fragment, the tail of a bubble): the director skips them
        entry["tiny"] = min(p["width"], p["height"]) < 120 or p["width"] * p["height"] < 40000
        panels.append(entry)

    bubbles, bubble_images = [], []
    for b in info["bubbles"]:
        im = Image.open(layers / b["file"]).convert("RGBA")
        im.save(out / f"{b['id']}.webp", quality=WEBP["bubble"], method=6)
        # the panel it belongs to: the one it sits in, or else the nearest one above or below (never a tiny one)
        cx, cy = b["x"] + b["width"] / 2, b["y"] + b["height"] / 2
        real = [p for p in panels if not p["tiny"]] or panels
        owner = next((p for p in real if p["y"] <= cy <= p["y"] + p["height"] and p["x"] <= cx <= p["x"] + p["width"]), None)
        between = owner is None
        if between:
            owner = min(real, key=lambda p: min(abs(cy - p["y"]), abs(cy - (p["y"] + p["height"]))))
        entry = {
            "id": b["id"], "panel": owner["id"],
            "x": b["x"] - owner["x"], "y": b["y"] - owner["y"], "w": b["width"], "h": b["height"],  # in the panel's pixels
            "between_panels": between,
            # small, flat "bubbles" are often eyes or highlights: the director checks these
            "suspect": b["width"] * b["height"] < 12000 or min(b["width"], b["height"]) < 70,
        }
        bubbles.append(entry)
        bubble_images.append((entry, im))

    pages_sheet(images, dropped, out / "sheets" / "pages.jpg")
    panel_sheets(panel_images, out / "sheets")
    bubbles_sheet(bubble_images, out / "sheets" / "bubbles.jpg")

    chapter = {
        "id": cid, "number": number, "source": cbz.name,
        "pages": len(images), "dropped_pages": {f"{k:03d}": why for k, why in sorted(dropped.items())},
        "strip": {"width": info["width"], "height": info["height"]},
        "panels": panels, "bubbles": bubbles,
        "webp_quality": WEBP,
        "seconds": {"split": round(split_seconds, 1), **steps},
    }
    (out / "chapter.json").write_text(json.dumps(chapter, indent=2))

    # 5. one zip to send to the director. Built locally, then moved into <out> whole, so a disconnect
    # halfway never leaves a broken zip that looks finished.
    local_zip = work / f"chapter_{cid}.zip"
    with zipfile.ZipFile(local_zip, "w", zipfile.ZIP_STORED) as z:  # WebP and JPEG are already compressed
        for f in sorted(out.rglob("*")):
            if f.is_file():
                z.write(f, f"{cid}/{f.relative_to(out)}")
    zip_path = out_root / local_zip.name
    partial = out_root / (local_zip.name + ".part")
    shutil.copyfile(local_zip, partial)
    partial.replace(zip_path)
    shutil.rmtree(work, ignore_errors=True)
    size = zip_path.stat().st_size / 1e6
    minutes = (time.time() - started) / 60
    tiny = sum(p["tiny"] for p in panels)
    print(f"  done: {len(panels)} panels ({tiny} tiny), {len(bubbles)} bubbles -> {zip_path.name} ({size:.1f} MB) in {minutes:.1f} min")
    if steps:  # where the time went, per panel
        other = split_seconds - sum(steps.values())
        print("  time per panel: " + ", ".join(f"{k} {v / len(panels):.1f} s" for k, v in steps.items())
              + f", saving and the rest {other / len(panels):.1f} s")
    return zip_path


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--cbz-dir", required=True, help="the folder with the chapter CBZ files")
    parser.add_argument("--out", required=True, help="where the finished chapters go")
    parser.add_argument("--chapters", required=True, help='which chapters: "94", "1-5", "1-5,94"')
    parser.add_argument("--skip", default="{}", help='pages to drop by hand, per chapter: \'{"94": [0, 69]}\' (-1 is the last page)')
    parser.add_argument("--work", default="work", help="a scratch folder (default: work)")
    parser.add_argument("--redo", action="store_true", help="make chapters again even if their zip already exists")
    parser.add_argument("--models", help="cutout models, as in split_layers.py (default: its choice)")
    parser.add_argument("--shard", default="1/1", help="K/N: take chapters K, K+N, K+2N, ... of the batch (default 1/1: all)")
    parser.add_argument("--done-dir", action="append", default=[], help="skip chapters whose zip is already here (repeatable)")
    parser.add_argument("--max-hours", type=float, help="do not start another chapter after this many hours")
    args = parser.parse_args()
    started = time.time()
    share, shares = (int(x) for x in args.shard.split("/"))
    if not 1 <= share <= shares:
        raise SystemExit(f"--shard {args.shard}: K must be between 1 and N")

    wanted = parse_chapters(args.chapters)
    skip = {str(k): v for k, v in json.loads(args.skip).items()}
    cbz_dir, out_root, work_root = Path(args.cbz_dir), Path(args.out), Path(args.work)
    out_root.mkdir(parents=True, exist_ok=True)

    found = {}
    # also in subfolders, in case the files were uploaded inside a folder of their own
    for f in sorted(cbz_dir.rglob("*"), key=lambda p: natural_key(p.name)):
        if f.suffix.lower() in {".cbz", ".zip"} and f.is_file():
            n = chapter_number(f)
            if n is not None and wanted(n):
                found.setdefault(n, f)
    if not found:
        raise SystemExit(f"No CBZ files for chapters {args.chapters} in {cbz_dir}.")
    # this run's share of the batch, fixed by position in the whole batch, so it is the same on every run
    found = {c: found[c] for i, c in enumerate(sorted(found, key=float)) if i % shares == share - 1}
    if shares > 1:
        print(f"Share {share} of {shares}: chapters {', '.join(sorted(found, key=float))}")
    done_dirs = [out_root] + [Path(d) for d in args.done_dir]
    is_done = lambda c: any(next(Path(d).rglob(f"chapter_{chapter_id(c)}.zip"), None) for d in done_dirs if Path(d).is_dir())
    todo = {c: f for c, f in found.items() if args.redo or not is_done(c)}
    for c in sorted(set(found) - set(todo), key=float):
        print(f"Chapter {c}: already done (chapter_{chapter_id(c)}.zip); use --redo to make it again")
    if not todo:
        return

    import split_layers
    models = split_layers.Models(args.models.split(",") if args.models else None)
    failed, left, longest = [], [], 0.0
    for c in sorted(todo, key=float):
        if args.max_hours is not None:
            # stop in time: not after the limit, and not if the slowest chapter so far would not fit
            remaining = args.max_hours * 3600 - (time.time() - started)
            if remaining < 1.2 * longest or remaining <= 0:
                left.append(c)
                continue
        print(f"Chapter {c}: {todo[c].name}")
        t = time.time()
        try:
            make_chapter(todo[c], c, out_root, work_root, models, skip.get(c, []))
        except Exception as e:  # keep going with the rest of the batch
            print(f"  FAILED: {e}")
            failed.append(c)
        longest = max(longest, time.time() - t)
    if left:
        print(f"Out of time: chapters {', '.join(left)} are left for the next run.")
    if failed:
        print(f"Failed: chapters {', '.join(failed)}. The others are done.")
        sys.exit(1)
    print(f"All done. Chapters are in {out_root.resolve()}")


if __name__ == "__main__":
    main()


### Step 5: make the chapters
The first lines say whether it runs on the graphics card (`Cutouts run on the graphics card (fast)`). The first run also downloads the AI models (about 2.3 GB).

In [ ]:
import json
skip = json.dumps({str(k): v for k, v in SKIP_PAGES.items()})
!python make_chapter.py --cbz-dir "{CBZ}" --out "{OUT}" --chapters "{CHAPTERS}" --skip '{skip}' --work /content/work {"--redo" if REDO else ""}

### Step 6: check the pages and download
Each chapter's pages are shown below. Pages crossed out in red were dropped: if a story page was dropped
(or a title or promo page was kept), set `SKIP_PAGES` in Step 1, set `REDO = True` and Run all again.

In [ ]:
import io, zipfile
from IPython.display import display
from PIL import Image
from make_chapter import parse_chapters, chapter_number
wanted = parse_chapters(CHAPTERS)
done = sorted(f for f in os.listdir(OUT) if f.startswith('chapter_') and f.endswith('.zip'))
ready = []
for f in done:
    with zipfile.ZipFile(f'{OUT}/{f}') as z:
        info = json.loads(z.read(next(n for n in z.namelist() if n.endswith('/chapter.json'))))
        if not wanted(info['number']):
            continue
        ready.append(f)
        size = os.path.getsize(f'{OUT}/{f}') / 1e6
        print(f"Chapter {info['number']}: {len(info['panels'])} panels, {len(info['bubbles'])} bubbles, "
              f"dropped pages {list(info['dropped_pages']) or 'none'}  ({f}, {size:.1f} MB)")
        sheet = Image.open(io.BytesIO(z.read(next(n for n in z.namelist() if n.endswith('sheets/pages.jpg')))))
        display(sheet.resize((1200, round(sheet.height * 1200 / sheet.width))))
if not ready:
    raise SystemExit('No finished chapters yet: scroll up to Step 5 to see what happened.')
print('Send these to Claude:', ', '.join(ready))
if DOWNLOAD:
    from google.colab import files
    for f in ready:
        files.download(f'{OUT}/{f}')
drive.flush_and_unmount()  # make sure everything is saved in Google Drive
print('Saved in Google Drive:', f'{DRIVE_FOLDER}/chapters')